In [2]:
from langchain import PromptTemplate                                    # PromptTemplate creation for llm model
from langchain.chains import RetrievalQA                                # Qustion answering behavours of LLM
from langchain.embeddings import HuggingFaceEmbeddings                  # Vector embading creation from text data
from langchain.vectorstores import Pinecone                             # Vector DB ("Knowledge base" and "semantic index")
import pinecone                                                         # Vector DB
from langchain.document_loaders import PyPDFLoader, DirectoryLoader     # Data injection from (dirctory and pdf)
from langchain.text_splitter import RecursiveCharacterTextSplitter      # Chunk creation from antire Corpus
from langchain.prompts import PromptTemplate                            # same PromptTemplate creation for llm model
from langchain.llms import CTransformers                                # Helpfull for work with "Quantize model"

#### Creating pinecone cluster ("semantic index")

In [3]:
# step1 - visite the site and login : https://app.pinecone.io/organizations/-NnYdPxXZp5EJezC195H/projects/gcp-starter:0zk0p7w/indexes
# Step2 - go to API key and (Default)copy : 19dff51a-bb17-4ab3-9103-d6342cd132e4
PINECONE_API_KEY = "19dff51a-bb17-4ab3-9103-d6342cd132e4"

# Create a indexes "https://app.pinecone.io/organizations/-NnYdPxXZp5EJezC195H/projects/gcp-starter:0zk0p7w/create-index" to get PINECONE_API_ENV
# name= "medical-chatbot", 
# Dimension=384 {vectorization model using :https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2, it gives out-put dimension 384}
# Metrics= cosine              
# Create index
PINECONE_API_ENV = "gcp-starter"  # ENVIRONMENT variable

In [4]:
#### Loding PDF file from data folder (Data injection)

def load_pdf(data):
    '''Extract data from the PDF'''
     
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()
    return documents

In [5]:
# Loding the pdf data into extracted_data
extracted_data = load_pdf("data/")
# extracted_data[:60]

In [8]:
#Create text chunks function
def text_split(extracted_data):
    '''Converting the PDF text into chunks'''
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20) 
    text_chunks = text_splitter.split_documents(extracted_data)      # spliting data
    return text_chunks

In [9]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 7020


In [11]:
#download embedding model
def download_hugging_face_embeddings():
    """converting all the Chunk text into vectors"""
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [12]:
embeddings = download_hugging_face_embeddings()

.gitattributes: 100%|██████████| 1.18k/1.18k [00:00<00:00, 75.7kB/s]
1_Pooling/config.json: 100%|██████████| 190/190 [00:00<00:00, 11.7kB/s]
README.md: 100%|██████████| 10.6k/10.6k [00:00<00:00, 451kB/s]
config_sentence_transformers.json: 100%|██████████| 116/116 [00:00<00:00, 7.45kB/s]
data_config.json: 100%|██████████| 39.3k/39.3k [00:00<00:00, 2.52MB/s]
pytorch_model.bin: 100%|██████████| 90.9M/90.9M [01:16<00:00, 1.18MB/s]
sentence_bert_config.json: 100%|██████████| 53.0/53.0 [00:00<?, ?B/s]
special_tokens_map.json: 100%|██████████| 112/112 [00:00<00:00, 7.16kB/s]
tokenizer.json: 100%|██████████| 466k/466k [00:00<00:00, 3.63MB/s]
tokenizer_config.json: 100%|██████████| 350/350 [00:00<?, ?B/s] 
train_script.py: 100%|██████████| 13.2k/13.2k [00:00<?, ?B/s]
vocab.txt: 100%|██████████| 232k/232k [00:00<00:00, 7.41MB/s]
modules.json: 100%|██████████| 349/349 [00:00<00:00, 36.6kB/s]


In [13]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={})

In [16]:
# --------------Testing the "embeddings" with some words ----------------------------------
query_result = embeddings.embed_query("Hello world") 
print(query_result)                     # Printing vector representation of "Hello world" using "all-MiniLM-L6-v2" model
print("Length", len(query_result))      # and the dimension was 384

[-0.03447727859020233, 0.03102315217256546, 0.006734953261911869, 0.02610895223915577, -0.03936200961470604, -0.16030248999595642, 0.06692399084568024, -0.006441458594053984, -0.04745051637291908, 0.014758842997252941, 0.07087532430887222, 0.055527545511722565, 0.019193314015865326, -0.02625134028494358, -0.010109533555805683, -0.026940464973449707, 0.022307470440864563, -0.022226575762033463, -0.1496926248073578, -0.0174929928034544, 0.007676254957914352, 0.0543522909283638, 0.003254421753808856, 0.03172592446208, -0.08462145179510117, -0.0294059906154871, 0.051595594733953476, 0.04812406003475189, -0.003314807778224349, -0.05827919393777847, 0.04196925461292267, 0.02221064642071724, 0.1281888782978058, -0.02233896031975746, -0.011656253598630428, 0.06292839348316193, -0.032876305282115936, -0.0912260115146637, -0.031175408512353897, 0.05269954353570938, 0.04703480377793312, -0.08420310914516449, -0.030056172981858253, -0.020744821056723595, 0.009517848491668701, -0.003721776884049177

In [17]:
#Initializing the Pinecone(V DB)
pinecone.init(api_key=PINECONE_API_KEY,
              environment=PINECONE_API_ENV)

index_name="medical-chatbot"

#Creating Embeddings for Each of The Text Chunks & storing
docsearch=Pinecone.from_texts([t.page_content for t in text_chunks], embeddings, index_name=index_name)

# IMP : # Now go to the Pinecone website and refresh the page to see the vectors
    # : Pinecone is a remote vector db if we wnat local vector Db then (Chroma DB) is a good choice

In [18]:
# ----------- Rank Results & Knowledge base modules ------------------
#If we already have an index we can load it like this
docsearch=Pinecone.from_existing_index(index_name, embeddings)

query = "What are Allergies"

docs=docsearch.similarity_search(query, k=3)   # It will give the Top 3 results 

print("Result", docs)
# Note : It doesnt give a appropriate answer to find the appropriate answer we need to give this promt and the Rank results to LLM to Get the Actual Out put. 

Result [Document(page_content="GALE ENCYCLOPEDIA OF MEDICINE 2 117Allergies\nAllergic rhinitis is commonly triggered by\nexposure to household dust, animal fur,or pollen. The foreign substance thattriggers an allergic reaction is calledan allergen.\nThe presence of an allergen causes the\nbody's lymphocytes to begin producingIgE antibodies. The lymphocytes of an allergy sufferer produce an unusuallylarge amount of IgE.\nIgE molecules attach to mast\ncells, which contain histamine.HistaminePollen grains\nLymphocyte\nFIRST EXPOSURE", metadata={}), Document(page_content='allergens are the following:\n• plant pollens\n• animal fur and dander\n• body parts from house mites (microscopic creatures\nfound in all houses)\n• house dust• mold spores• cigarette smoke• solvents• cleaners\nCommon food allergens include the following:\n• nuts, especially peanuts, walnuts, and brazil nuts\n• fish, mollusks, and shellfish• eggs• wheat• milk• food additives and preservatives\nThe following types of drug

In [19]:
# -------- Defining Prompt Template -------------------------
prompt_template="""
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [20]:
# chain type Promt template wor "Retrival QA chain"
PROMPT=PromptTemplate(template=prompt_template, input_variables=["context", "question"])
chain_type_kwargs={"prompt": PROMPT}

In [22]:
#---------------- Defining LLM (llama-2-7b-chat.ggmlv3.q4_0.bin) model ------------------------------------------
llm=CTransformers(model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",
                  model_type="llama",
                  config={'max_new_tokens':512,
                          'temperature':0.8})

In [24]:
# Assigning a retrival QA chain from langchain
qa=RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=docsearch.as_retriever(search_kwargs={'k': 2}), # It will give two relavent answer
    return_source_documents=True,                             # From that only give me the true relavent answer
    chain_type_kwargs=chain_type_kwargs)

In [26]:
# loop for taking input from users 

while True:
    user_input=input(f"Input Prompt:")
    result=qa({"query": user_input})
    print("Response : ", result["result"])

Response :  Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.
